# REF4-126 strict-forward GPU runner (Colab T4)

This notebook is the execution wrapper for `REF4-JM-R-RESIDUAL-STRICT-GPU-126`. It mounts Google Drive, verifies every staged input against `SHA256SUMS`, requires a CUDA T4 by default, materializes the code bundle under `/content`, and streams the runner log and checkpoints back to Drive.

Expected Drive layout:

```text
REF4_126/
├── SHA256SUMS
├── data/
│   ├── train.csv
│   └── trackman_history.csv
├── anchor/strict_113A/oof_predictions.csv
├── code/
│   ├── REF4_126_CODE.zip        # preferred
│   └── ...                       # or an unpacked repo containing scripts/
├── checkpoints/                  # created here; reused with --resume
├── logs/                         # created here
└── results/                      # created here
```

For an ordinary folder shared with your Google account, add a shortcut to My Drive and put its mounted path in `DRIVE_ROOT_TEXT`. A Drive folder ID is not a filesystem path; the shortcut gives it a stable path without putting rclone or Drive API credentials in Colab. Do not run two writers against the same checkpoint directory.

Provisional runner CLI contract (the runner was not present when this notebook was created):

```text
python scripts/run_ref4_jm_r_residual_strict_gpu_126.py \
  --data-dir <REF4_126/data> \
  --output-dir <REF4_126/results> \
  --checkpoint-dir <REF4_126/checkpoints> \
  --anchor-oof <REF4_126/anchor/strict_113A/oof_predictions.csv> \
  --targets 2022,2023,2024 --device gpu --resume
```

The runner must return nonzero on failure and write `results/result.json` with `status` equal to `COMPLETE` on success. Checkpoints should be written atomically before a fold is considered resumable.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import datetime as dt
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time


In [ ]:
#@title 1. Paths and run settings
DRIVE_ROOT_TEXT = '/content/drive/MyDrive/LG aimer/REF4_126' #@param {type:'string'}
TARGETS = '2022,2023,2024' #@param {type:'string'}
REQUIRE_T4 = True #@param {type:'boolean'}
INSTALL_REQUIREMENTS = True #@param {type:'boolean'}
RESUME = True #@param {type:'boolean'}

DRIVE_ROOT = Path(DRIVE_ROOT_TEXT).expanduser()
DATA_DIR = DRIVE_ROOT / 'data'
ANCHOR_OOF = DRIVE_ROOT / 'anchor' / 'strict_113A' / 'oof_predictions.csv'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
LOG_DIR = DRIVE_ROOT / 'logs'
RESULT_DIR = DRIVE_ROOT / 'results'
MANIFEST_PATH = DRIVE_ROOT / 'SHA256SUMS'
RUNTIME_ROOT = Path('/content/ref4_126_runtime')

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(
        f'Drive root not found: {DRIVE_ROOT}. Add the shared-folder shortcut to My Drive '
        'or update DRIVE_ROOT_TEXT to its mounted path.'
    )
for path in (CHECKPOINT_DIR, LOG_DIR, RESULT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print('Drive root :', DRIVE_ROOT)
print('Data       :', DATA_DIR)
print('Anchor OOF :', ANCHOR_OOF)
print('Checkpoints:', CHECKPOINT_DIR)
print('Results    :', RESULT_DIR)


In [ ]:
#@title 2. Verify staged inputs against SHA256SUMS
def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def parse_sha256sums(path: Path) -> dict[str, str]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing checksum manifest: {path}')
    entries = {}
    for line_number, raw in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split(maxsplit=1)
        if len(parts) != 2 or len(parts[0]) != 64:
            raise ValueError(f'Invalid SHA256SUMS line {line_number}: {raw!r}')
        expected, relative_text = parts
        relative_text = relative_text.lstrip('*')
        relative = Path(relative_text)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError(f'Unsafe manifest path at line {line_number}: {relative}')
        if relative.as_posix() in entries:
            raise ValueError(f'Duplicate manifest entry: {relative}')
        entries[relative.as_posix()] = expected.lower()
    if not entries:
        raise ValueError('SHA256SUMS contains no file entries')
    return entries

manifest = parse_sha256sums(MANIFEST_PATH)
required_data = {
    'data/train.csv', 'data/trackman_history.csv',
    'anchor/strict_113A/oof_predictions.csv',
}
missing_manifest_entries = sorted(required_data - manifest.keys())
if missing_manifest_entries:
    raise RuntimeError(f'Required data are not bound by SHA256SUMS: {missing_manifest_entries}')

verified = []
for relative, expected in manifest.items():
    source = DRIVE_ROOT / relative
    if not source.is_file():
        raise FileNotFoundError(f'Manifest file missing from Drive: {source}')
    actual = sha256_file(source)
    if actual != expected:
        raise RuntimeError(f'SHA-256 mismatch: {relative}: {actual} != {expected}')
    verified.append({'path': relative, 'bytes': source.stat().st_size, 'sha256': actual})
    print(f'OK {actual[:12]}  {relative}  ({source.stat().st_size / 1024**2:.1f} MiB)')

print(f'Verified {len(verified)} staged files.')


In [ ]:
#@title 3. Require a usable CUDA T4
query = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    text=True, capture_output=True, check=False,
)
if query.returncode != 0:
    raise RuntimeError(
        'nvidia-smi failed. In Colab choose Runtime > Change runtime type > T4 GPU.\n'
        + query.stderr
    )
gpu_lines = [line.strip() for line in query.stdout.splitlines() if line.strip()]
if len(gpu_lines) != 1:
    raise RuntimeError(f'Expected exactly one GPU, got: {gpu_lines}')
gpu_name = gpu_lines[0].split(',', 1)[0].strip()
if REQUIRE_T4 and 'T4' not in gpu_name.upper():
    raise RuntimeError(f'This run requires a T4, but Colab assigned: {gpu_name}')
print('GPU:', gpu_lines[0])
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
#@title 4. Materialize the verified code bundle under /content
def safe_extract_zip(archive_path: Path, destination: Path) -> None:
    import zipfile
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or '..' in member_path.parts:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
            target = (destination / member_path).resolve()
            if destination_resolved not in (target, *target.parents):
                raise RuntimeError(f'ZIP member escapes destination: {member.filename}')
        archive.extractall(destination)

code_zip_candidates = [
    DRIVE_ROOT / 'code' / 'REF4_126_CODE.zip',
    DRIVE_ROOT / 'REF4_126_CODE.zip',
]
verified_paths = set(manifest)
code_zip = next((path for path in code_zip_candidates if path.is_file()), None)
unpacked_code = DRIVE_ROOT / 'code'

if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
RUNTIME_ROOT.mkdir(parents=True)

if code_zip is not None:
    relative_zip = code_zip.relative_to(DRIVE_ROOT).as_posix()
    if relative_zip not in verified_paths:
        raise RuntimeError(f'Code archive is not bound by SHA256SUMS: {relative_zip}')
    safe_extract_zip(code_zip, RUNTIME_ROOT)
elif (unpacked_code / 'scripts').is_dir():
    runner_relative = 'code/scripts/run_ref4_jm_r_residual_strict_gpu_126.py'
    if runner_relative not in verified_paths:
        raise RuntimeError(
            'Unpacked runner must be listed in SHA256SUMS. Prefer code/REF4_126_CODE.zip.'
        )
    shutil.copytree(unpacked_code, RUNTIME_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError(
        'No code bundle found. Expected code/REF4_126_CODE.zip or code/scripts/.'
    )

runner_matches = list(RUNTIME_ROOT.rglob('scripts/run_ref4_jm_r_residual_strict_gpu_126.py'))
if len(runner_matches) != 1:
    raise RuntimeError(f'Expected one runner in the code bundle, found {runner_matches}')
RUNNER = runner_matches[0]
CODE_ROOT = RUNNER.parent.parent
print('Code root:', CODE_ROOT)
print('Runner   :', RUNNER)


In [ ]:
#@title 5. Install pinned runner dependencies and validate CUDA packages
requirements_candidates = [
    CODE_ROOT / 'requirements-colab.txt',
    CODE_ROOT / 'requirements.txt',
]
requirements_path = next((path for path in requirements_candidates if path.is_file()), None)
if INSTALL_REQUIREMENTS:
    if requirements_path is None:
        print('No requirements file in code bundle; installing the expected CatBoost build only.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'catboost==1.2.10'], check=True)
    else:
        print('Installing:', requirements_path)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)], check=True)

import catboost
import torch
print('Python  :', sys.version.split()[0])
print('CatBoost:', catboost.__version__)
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.version.cuda, torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot see CUDA after dependency installation.')
print('Torch GPU:', torch.cuda.get_device_name(0))


In [ ]:
#@title 6. Launch strict-forward training and tee logs to Drive
def atomic_write_json(path: Path, payload: dict) -> None:
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

timestamp = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
log_path = LOG_DIR / f'ref4_126_{timestamp}.log'
state_path = LOG_DIR / 'last_launch.json'
command = [
    sys.executable, str(RUNNER),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(RESULT_DIR),
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--anchor-oof', str(ANCHOR_OOF),
    '--targets', TARGETS,
    '--device', 'gpu',
    '--no-cpu-fallback',
]
if RESUME:
    command.append('--resume')

launch = {
    'experiment_id': 'REF4-JM-R-RESIDUAL-STRICT-GPU-126',
    'status': 'RUNNING',
    'started_utc': timestamp,
    'command': command,
    'drive_root': str(DRIVE_ROOT),
    'code_root': str(CODE_ROOT),
    'runner_sha256': sha256_file(RUNNER),
    'gpu': gpu_lines[0],
    'verified_inputs': verified,
    'log': str(log_path),
}
atomic_write_json(state_path, launch)
print('Command:', ' '.join(command))
print('Log    :', log_path)

started = time.monotonic()
env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
with log_path.open('a', encoding='utf-8') as log_file:
    log_file.write('=== COMMAND ===\n' + ' '.join(command) + '\n')
    log_file.flush()
    process = subprocess.Popen(
        command, cwd=CODE_ROOT, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    returncode = process.wait()

launch.update({
    'status': 'COMPLETE' if returncode == 0 else 'FAILED',
    'returncode': returncode,
    'elapsed_seconds': time.monotonic() - started,
    'finished_utc': dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ'),
})
atomic_write_json(state_path, launch)
print(json.dumps({key: launch[key] for key in ('status', 'returncode', 'elapsed_seconds')}, indent=2))
if returncode != 0:
    tail = '\n'.join(log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-120:])
    print('===== LOG TAIL =====\n' + tail)
    raise RuntimeError(f'REF4-126 runner failed with return code {returncode}; resume after fixing the error.')


In [ ]:
#@title 7. Validate and display persisted results
result_path = RESULT_DIR / 'result.json'
if not result_path.is_file():
    raise FileNotFoundError(
        f'Runner exited successfully but did not produce the required result contract: {result_path}'
    )
result = json.loads(result_path.read_text(encoding='utf-8'))
if result.get('status') != 'COMPLETE':
    raise RuntimeError(f"result.json status is not COMPLETE: {result.get('status')!r}")
print(json.dumps(result, ensure_ascii=False, indent=2))

try:
    import pandas as pd
    metric_candidates = sorted(RESULT_DIR.glob('*metric*.csv'))
    for metric_path in metric_candidates:
        print('Metrics:', metric_path.name)
        display(pd.read_csv(metric_path))
except Exception as error:
    print('Metric preview skipped:', repr(error))

print('Checkpoint files:')
for path in sorted(CHECKPOINT_DIR.rglob('*')):
    if path.is_file():
        print(f'{path.relative_to(CHECKPOINT_DIR)}  {path.stat().st_size / 1024**2:.1f} MiB')


## Resume procedure

If the Colab VM disconnects, reconnect to a T4 and run the notebook from the top with the same `DRIVE_ROOT_TEXT`, `TARGETS`, and `RESUME=True`. The runtime code directory is intentionally rebuilt from the verified Drive bundle; only Drive checkpoints are trusted across sessions.

If a checkpoint was interrupted while being written, preserve it by renaming it to `*.corrupt.<UTC timestamp>` and rerun. Do not delete or overwrite verified inputs, and do not launch another notebook against the same `checkpoints/` directory.
